# Machine Learning for Applied Research

## 7: Cross-Validation & Generalizability

In this notebook, we will cover:
- Why a single train/test split can be misleading, and what k-fold cross-validation does about it
- Revisiting overfitting and underfitting through cross-validation instead of a single split
- Data leakage: what it is, and common ways it sneaks into a research pipeline

### Introduction

We previously used a single train/test split throughout. But a single split is itself a source of uncertainty: a different random split could give a meaningfully different result, especially with a dataset as small as our 768-row Energy Efficiency data, or the even smaller datasets many of you work with in your own research. Cross-validation addresses this directly. This notebook covers k-fold CV, revisits over/underfitting through that more reliable lens, and then covers data leakage, one of the most common and damaging mistakes in applied ML research.

### Why One Split Isn't Enough

K-fold cross-validation splits the data into k roughly equal parts (folds). It then trains and evaluates the model k times, each time holding out a different fold as the test set and training on the remaining k-1 folds. The result is k performance scores rather than one - averaging them gives a more reliable estimate of performance, and their spread tells you how much that estimate varies.

In [ ]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

energy = fetch_ucirepo(id=242)
X = energy.data.features
y = energy.data.targets

X.columns = ["Relative Compactness",
             "Surface Area",
             "Wall Area",
             "Roof Area",
             "Overall Height",
             "Orientation",
             "Glazing Area",
             "Glazing Area Distribution"]

y.columns = ["Heating Load", "Cooling Load"]

target_col = y.columns[0]   # Heating Load
df = pd.concat([X, y[[target_col]]], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[target_col]), df[target_col], test_size=0.25, random_state=42
)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_num, (train_idx, val_idx) in enumerate(kf.split(X_train), start=1):
    print(f"Fold {fold_num}: train={len(train_idx)} rows, validation={len(val_idx)} rows")

In practice, you rarely loop through folds by hand - `cross_val_score` does it for you:

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    LinearRegression(), X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
)
print(f"CV RMSE per fold: {-cv_scores}")
print(f"Mean CV RMSE: {-cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})")

(The score is negated because scikit-learn's scoring convention always treats higher as better, and RMSE is something we want to minimize - hence "neg_root_mean_squared_error".)

#### 1.1 Exercises:
> (a) Run `cross_val_score` the same way, but for a `RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)`.
>
> (b) Print the mean and standard deviation of the CV RMSE for both the linear regression above and your Random Forest.
>
> (c) Which model has the lower mean CV RMSE? Which has the more consistent (lower standard deviation) performance across folds? Are those necessarily the same model?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
forest_cv_scores = cross_val_score(
    RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42),
    X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
)

print(f"Linear Regression - Mean CV RMSE: {-cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})")
print(f"Random Forest      - Mean CV RMSE: {-forest_cv_scores.mean():.2f} (+/- {forest_cv_scores.std():.2f})")

# (c) The Random Forest typically has both the lower mean CV RMSE and the
# lower standard deviation here, but these are two different questions
# (how good on average, vs. how consistent), and they don't have to point
# to the same model. A model could have excellent average performance but
# be unstable across folds (a warning sign for how it might behave on
# genuinely new data), or vice versa.

#### 1.2 Exercises:
> (a) Run `cross_val_score` for the Random Forest (from Exercise 1.1) three times, with `cv=3`, `cv=5`, and `cv=10`, and print the mean and standard deviation of CV RMSE each time.
>
> (b) Does the mean CV RMSE change much across the three settings? Does the standard deviation?
>
> (c) More folds means more training runs (`cv=10` fits the model 10 times instead of 3). In your own research, with a dataset of a given size, what would push you toward more folds vs. fewer?

In [ ]:
# Your solution:





##### Solution:

In [ ]:
for k in [3, 5, 10]:
    scores_k = cross_val_score(
        RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42),
        X_train, y_train, cv=k, scoring="neg_root_mean_squared_error"
    )
    print(f"cv={k:2d} - Mean RMSE: {-scores_k.mean():.2f} (+/- {scores_k.std():.2f})")

# (c) More folds means each training set is larger (closer to the full
# dataset) and the estimate is averaged over more runs, which usually
# reduces variance in the estimate itself. This is valuable with a small
# dataset where every row matters. The cost is purely computational:
# more folds means more model fits. With a small, cheap-to-train model
# like this, that trade-off is easy; with a slow-to-train model or a
# very large dataset, fewer folds (or a single validation split) may be
# the more practical choice.

### Overfitting and Underfitting, Revisited

Notebook 4 used `max_depth` for a decision tree using a single train/test split. Let's redo that comparison properly, using cross-validation instead of one split for each depth as a more trustworthy way to find where a model starts to overfit.

In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeRegressor

depth_range = range(1, 16)

train_scores, val_scores = validation_curve(
    DecisionTreeRegressor(random_state=42), X_train, y_train,
    param_name="max_depth", param_range=depth_range,
    cv=5, scoring="neg_root_mean_squared_error"
)

train_rmse_mean = -train_scores.mean(axis=1)
val_rmse_mean = -val_scores.mean(axis=1)

`validation_curve` repeats the same 5-fold cross-validation at every value of `max_depth` in `depth_range`, and returns the train and validation scores from every fold at every depth - `train_scores` and `val_scores` are both arrays of shape `(len(depth_range), 5)`, which is why we average across the folds (`axis=1`) above.

#### 1.3 Exercises:
> (a) Plot `train_rmse_mean` and `val_rmse_mean` against `depth_range` on one chart (`import matplotlib.pyplot as plt`).
>
> (b) Identify which depth has the lowest mean cross-validated RMSE.
>
> (c) Compare this to the "best depth" you found in Notebook 4 using a single train/test split. Are they the same? If not, which estimate would you trust more, and why?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(list(depth_range), train_rmse_mean, marker="o", label="Train RMSE")
plt.plot(list(depth_range), val_rmse_mean, marker="o", label="Cross-validated RMSE")
plt.xlabel("max_depth")
plt.ylabel("RMSE")
plt.title("Decision tree: train vs. cross-validated RMSE by depth")
plt.legend()
plt.show()

best_depth_cv = list(depth_range)[int(np.argmin(val_rmse_mean))]
print(f"Depth with lowest cross-validated RMSE: {best_depth_cv}")

# (c) The cross-validated best depth may or may not match Notebook 4's
# single-split answer: with a dataset this size, a single split's "best
# depth" can easily be a bit of noise from that particular split. The
# cross-validated estimate, averaged over 5 different train/validation
# partitions, is the one worth trusting more: it is far less sensitive to
# which specific rows happened to land in the test set.

Now that we have a depth we trust, it's worth actually looking at the tree itself. Scikit-learn's `plot_tree` renders the full split structure.

In [ ]:
from sklearn.tree import plot_tree

best_tree_cv = DecisionTreeRegressor(max_depth=best_depth_cv, random_state=42)
best_tree_cv.fit(X_train, y_train)

plt.figure(figsize=(16, 8))
plot_tree(best_tree_cv, feature_names=X_train.columns, filled=True, fontsize=8)
plt.title(f"Decision tree at max_depth={best_depth_cv} (cross-validated best)")
plt.show()

Each box shows the split question, how many training rows reached that point, and the average heating load among them, color shows the same average value, so you can trace which branches lead toward high vs. low predictions. If `best_depth_cv` came out fairly large, the plot below will be dense; the split at the very top is the single most useful question the tree found, and is worth reading even if the lower branches are hard to make out at this size.

Pick one split near the top of the tree and explain in your own words what threshold question it is asking, and why that might be a sensible first question to ask about a building's heating load.

### Avoiding Data Leakage

Data leakage happens whenever information that would not genuinely be available at prediction time (often information from the test set, or from the future, or from a variable downstream of the target) accidentally influences training. The result is a performance estimate that looks great on paper and does not hold up in practice. A few common forms:

- **Preprocessing leakage**: fitting a scaler, imputer, or feature selector on the *entire* dataset before splitting, so it "sees" the test rows' statistics.
- **Duplicate leakage**: near-duplicate or repeated-measurement rows split across train and test, letting the model partly recognize a specific specimen rather than generalize.
- **Temporal leakage**: using information from the future to predict the past, which poses a real risk once we start forecasting with the Bike Sharing dataset in the next notebook.
- **Target leakage**: including a feature that is itself a downstream consequence of the target (why Notebook 6 excluded the specific failure-mode columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) from the AI4I features). Those columns are only set once a failure has already happened, so including them would have let the model "cheat."

In [ ]:
from sklearn.preprocessing import StandardScaler

# WRONG: fit the scaler on the full dataset, before splitting
scaler_leaky = StandardScaler()
scaler_leaky.fit(df.drop(columns=[target_col]))

# RIGHT: fit the scaler only on the training data
scaler_correct = StandardScaler()
scaler_correct.fit(X_train)

comparison = pd.DataFrame({
    "Mean (leaky, all 768 rows)": scaler_leaky.mean_,
    "Mean (correct, 576 train rows)": scaler_correct.mean_,
}, index=X_train.columns)

print(comparison)

The two columns are not identical, the "leaky" scaler has quietly used information from the 192 rows that are supposed to be unseen test data. On its own this looks like a small numerical difference, but the same principle applies to any preprocessing step, and the cumulative effect across a whole pipeline can meaningfully inflate reported performance. The upcoming pipelines notebook covers the systematic fix: `Pipeline` objects that guarantee every preprocessing step is refit inside each cross-validation fold, never on the full dataset.

#### 1.4 Exercises:
> For each scenario below, decide whether it involves data leakage, and if so, briefly describe the mechanism:
>
> (a) A researcher scales all features using the full dataset's mean and standard deviation, then splits into train and test.
>
> (b) A researcher measures the same physical specimen five times and puts some of those repeated measurements in the training set and others in the test set.
>
> (c) A researcher predicts next month's energy demand using only data available up to today, with a train/test split that respects time order (all training data comes before all test data).
>
> (d) A researcher predicting machine failure includes a feature recording "total maintenance downtime for this event," which is only ever recorded once a failure has already happened.


In [ ]:
# Your solution:





##### Solution:


In [ ]:
# (a) Leakage. The scaler's mean and standard deviation are computed
#     using information from what should be unseen test rows, before the
#     split even happens.

# (b) Leakage. Repeated measurements of the same specimen are correlated
#     with each other, if some measurements from a given specimen are in
#     training and others in test, the model can partly "recognize" that
#     specimen rather than genuinely generalize to new ones.

# (c) Not leakage. This is the correct way to handle data with a
#     meaningful time order: train strictly on the past, test on data
#     that comes strictly after it.

# (d) Leakage (target leakage). "Total maintenance downtime for this
#     event" only exists because a failure already happened. It is a
#     downstream consequence of the target, not a genuine predictor
#     available before a failure occurs. Including it would let the model
#     effectively cheat.

#### 1.5 Exercises:
> Below is a short workflow someone on your team proposes. Find the leak.
>
> ```python
> from sklearn.preprocessing import StandardScaler
> from sklearn.model_selection import train_test_split
> from sklearn.linear_model import Ridge
>
> scaler = StandardScaler()
> X_scaled = scaler.fit_transform(X)          
> X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.25, random_state=0)
>
> model = Ridge().fit(X_tr, y_tr)
> print(model.score(X_te, y_te))
> ```
>
> (a) Identify exactly which line causes the leak.
>
> (b) Rewrite the snippet so the scaler is fit correctly.
>
> (c) Would this leak tend to make the reported test score look better or worse than the model's true performance? Why?

In [ ]:
# Your solution:





##### Solution:


```python
# (a) The leak is on the `scaler.fit_transform(X)` line, it fits the
# scaler on the full dataset, including rows that end up in the test
# set, before the split ever happens.

# (b) Corrected version:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)   # fit only on train
X_te_scaled = scaler.transform(X_te)       # transform only, never refit

model = Ridge().fit(X_tr_scaled, y_tr)
print(model.score(X_te_scaled, y_te))

# (c) This kind of leak tends to make the reported test score look
# better than the model's true performance, since the "test" data
# already influenced how every feature was rescaled - the evaluation is
# no longer on genuinely unseen data in the strictest sense.
```

#### Challenge:
> Run 5-fold cross-validation on the AI4I classifier (`RandomForestClassifier(random_state=42)`) using `StratifiedKFold` instead of plain `KFold`, scoring on `"roc_auc"`. Why is `StratifiedKFold` especially important for this dataset, compared to Energy Efficiency?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

maintenance = fetch_ucirepo(id=601)
feature_cols = ["Type", "Air temperature", "Process temperature",
                 "Rotational speed", "Torque", "Tool wear"]
maint_df = pd.concat(
    [maintenance.data.features[feature_cols], maintenance.data.targets[["Machine failure"]]],
    axis=1
)
maint_df["Type"] = maint_df["Type"].map({"L": 0, "M": 1, "H": 2})
X_maint = maint_df.drop(columns=["Machine failure"])
y_maint = maint_df["Machine failure"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf_cv_scores = cross_val_score(
    RandomForestClassifier(random_state=42), X_maint, y_maint, cv=skf, scoring="roc_auc"
)
print(f"Stratified CV ROC AUC per fold: {clf_cv_scores}")
print(f"Mean: {clf_cv_scores.mean():.3f} (+/- {clf_cv_scores.std():.3f})")

# With only about 3% positive examples, an ordinary KFold split could
# easily produce a fold with very few (or by bad luck, zero) failures
# in its validation portion, making that fold's score unreliable or even
# undefined. StratifiedKFold keeps roughly the same failure rate in every
# fold, the same idea behind stratify= in train_test_split from
# Notebook 6.

## Conclusion

This notebook made model evaluation considerably more rigorous:

- **K-fold cross-validation**: a more reliable performance estimate than a single train/test split, with a sense of how much that estimate varies, and how the choice of `k` trades estimate stability against compute cost
- **Overfitting and underfitting, revisited**: `validation_curve` lets you sweep a setting like `max_depth` using cross-validation instead of one split, correcting for noise a single split can introduce
- **Seeing the tree itself**: `plot_tree` turns the "recursively splitting on threshold questions" idea from Notebook 4 into something you can actually read
- **Data leakage**: preprocessing, duplicate, temporal, and target leakage, practiced both as scenario recognition and as spotting a leak hiding in real code
- **StratifiedKFold**: the classification analog of `stratify=` in `train_test_split`, essential for imbalanced data like AI4I